In [61]:
import pandas as pd
import numpy as np
import time
from tqdm import tqdm
import unidecode
import re
from rapidfuzz import fuzz
from collections import defaultdict

def normalize_name(name):
    if pd.isnull(name):
        return ""
    name = name.lower().strip()                      # Lowercase and trim
    name = unidecode.unidecode(name)                 # Remove accents
    name = name.replace("penal", "")                 # Remove common noise terms
    name = name.replace("centro", "")
    name = name.replace("cereso", "")
    name = name.replace("centro", "")
    name = name.replace("carcel", "")
    name = name.replace("distrital", "")
    name = name.replace("municipal", "")
    name = name.replace("reclusorio", "")
    name = name.replace("penitenciaria", "")
    name = name.replace("regional", "")
    name = name.replace("estatal", "")
    name = name.replace("cerereso", "")
    name = name.replace("establecimiento penitenciario", "")
    name = name.replace("cecjude", "")
    name = name.replace('*', '')
    name = ' '.join(name.split())                    # Remove extra spaces
    return name

def normalize_name2(name):
    if pd.isnull(name):
        return ""
    name = name.lower().strip()                      # Lowercase and trim
    name = unidecode.unidecode(name)                 # Remove accents
    name = name.replace("penal", "")                 # Remove common noise terms
    name = name.replace("centro", "")
    name = name.replace('*', '')
    name = ' '.join(name.split())                    # Remove extra spaces
    return name


In [ ]:
df2000 = pd.read_excel(f'../../../data/00-map/capacity/raw/2000/capacity_2_checked.xlsx')
df2006 = pd.read_excel(f'../../../data/00-map/capacity/raw/2006/capacity_2_checked.xlsx')
df2012 = pd.read_excel(f'../../../data/00-map/capacity/raw/2012/capacity_12_checked.xlsx')

# First deduplication

dfc = pd.concat([df2000[['center_name', 'year']], df2006[['center_name', 'year']], 
                                           df2012[['center_name', 'year']]])


dfc['name_clean'] = dfc['center_name'].apply(normalize_name2)

dfc['name_clean_cluster'] = dfc['center_name'].apply(normalize_name)

mexican_states = [
    "aguascalientes", "baja california", "baja california sur", "campeche", "distrito federal",
    "coahuila", "colima", "chiapas", "chihuahua", "durango", "guanajuato", "guerrero",
    "hidalgo", "jalisco", "mexico", "michoacan", "morelos", "nayarit", "nuevo leon",
    "oaxaca", "puebla", "queretaro", "quintana roo", "san luis potosi", "sinaloa",
    "sonora", "tabasco", "tamaulipas", "tlaxcala", "veracruz", "yucatan", "zacatecas"
]

state_codes = {
    "aguascalientes": "01",
    "baja california": "02",
    "baja california sur": "03",
    "campeche": "04",
    "coahuila": "05",
    "colima": "06",
    "chiapas": "07",
    "chihuahua": "08",
    "ciudad de mexico": "09",       # modern name
    "distrito federal": "09",       # legacy name, same code
    "durango": "10",
    "guanajuato": "11",
    "guerrero": "12",
    "hidalgo": "13",
    "jalisco": "14",
    "mexico": "15",
    "michoacan": "16",
    "morelos": "17",
    "nayarit": "18",
    "nuevo leon": "19",
    "oaxaca": "20",
    "puebla": "21",
    "queretaro": "22",
    "quintana roo": "23",
    "san luis potosi": "24",
    "sinaloa": "25",
    "sonora": "26",
    "tabasco": "27",
    "tamaulipas": "28",
    "tlaxcala": "29",
    "veracruz": "30",
    "yucatan": "31",
    "zacatecas": "32"
}

removal = [
    "instituciones penitenciarias federales", "cefereso", "ceferepsi", "islas marias", "federal femenil",
    "total"
]

removalre = "|".join([re.escape(unidecode.unidecode(state.lower())) for state in removal])

# Flag rows with unwanted text
dfc['remove'] = dfc['name_clean'].str.contains(removalre, regex=True)

# Filter them out
dfc = dfc[~dfc['remove']].copy()

dfc['is_state'] = dfc['name_clean'].isin(mexican_states)
dfc['state'] = dfc['name_clean'].where(dfc['is_state']).ffill()

dfc = dfc[~dfc['is_state']].drop(columns=['is_state', 'remove'])
dfc['CVE_ENT'] = dfc['state'].map(state_codes)
dfc

In [63]:
def cluster_prisons_within_state(state_df, threshold=90):
    names = state_df['name_clean_cluster'].unique()
    seen = set()
    clusters = []

    for name in names:
        if name in seen:
            continue
        cluster = [name]
        seen.add(name)
        for other in names:
            if other not in seen and fuzz.token_sort_ratio(name, other) >= threshold:
                cluster.append(other)
                seen.add(other)
        clusters.append(cluster)
    return clusters

state_clusters = {}

for state_code, group in dfc.groupby('CVE_ENT'):
    clusters = cluster_prisons_within_state(group)
    state_clusters[state_code] = clusters

name_map = {}

for state_code, clusters in state_clusters.items():
    for cluster in clusters:
        canonical = min(cluster, key=len)  # or use: Counter(df[df['prison_clean'].isin(cluster)]['prison_clean']).most_common(1)[0][0]
        for name in cluster:
            name_map[(state_code, name)] = canonical

dfc['prison_standardized'] = dfc.apply(lambda row: name_map.get((row['CVE_ENT'], row['name_clean_cluster']), 
                                                                row['name_clean_cluster']), axis=1)

## Manually clean the last ones:

dfc.drop_duplicates(['prison_standardized', 'CVE_ENT'])[['prison_standardized', 
                                            'CVE_ENT', 
                                            'state']].sort_values(by='CVE_ENT').to_excel('../../../data/00-map/capacity/geolocate/prison_stand.xlsx', 
                                                                                         index=False)

In [67]:
df_dedup = pd.read_excel('../../../data/00-map/capacity/geolocate/prison_stand_manual.xlsx')

dfcc = dfc.merge(df_dedup[['prison_standardized', 'cluster_manual']], on = 'prison_standardized', 
                how = 'left')

grouped = (
    dfcc.groupby(['CVE_ENT', 'state', 'cluster_manual'])['center_name']
    .unique()
    .reset_index()
)

# Convert the list of names to multiple columns: clean_name1, clean_name2, ...
max_names = grouped['center_name'].apply(len).max()

# Expand list to columns
names_df = pd.DataFrame(grouped['center_name'].tolist(), columns=[f'center_name{i+1}' for i in range(max_names)])

# Combine with the cluster info
dedup_clusters = pd.concat([grouped[['CVE_ENT', 'state', 'cluster_manual']], names_df], axis=1)
grouped = (
    dfcc.groupby(['CVE_ENT', 'state', 'cluster_manual'])['name_clean']
    .unique()
    .reset_index()
)

# Convert the list of names to multiple columns: clean_name1, clean_name2, ...
max_names = grouped['name_clean'].apply(len).max()

# Expand list to columns
names_df = pd.DataFrame(grouped['name_clean'].tolist(), columns=[f'name_clean{i+1}' for i in range(max_names)])

# Combine with the cluster info
dedup_clusters2 = pd.concat([grouped[['CVE_ENT', 'state', 'cluster_manual']], names_df], axis=1)

dedup_clusters = dedup_clusters.merge(dedup_clusters2, on = ['CVE_ENT', 'state', 'cluster_manual'], 
                                      how = 'left')
dedup_clusters.to_excel('../../../data/00-map/capacity/geolocate/deduplicated_prisons_2012.xlsx', 
                        index = False)